# Bài 5 · Series & DataFrame chuyên sâu

**Lập trình xử lý dữ liệu · 2627-1 · Viện TTNT, UET-VNU**

Chọn **Save a copy in Drive** trước khi sửa.

## Mục tiêu bài học

1. Giải thích cách phép tính và phép gán Series khớp theo nhãn.
2. Dùng hàm hoặc phép toán cột để thực hiện đúng quy tắc tính.
3. Phân biệt tổng hợp theo nhóm với kết quả cho từng hàng.
4. Dự đoán và kiểm tra số hàng sau khi ghép bảng.
5. Viết phân tích thành một chuỗi trả bảng mới để chạy lại luôn ra cùng kết quả.

Các bảng nhỏ dùng dữ liệu giả lập. Dữ liệu thật dùng Santiago ngày 29/06/2026, cùng snapshot của Bài 4. Môi trường kiểm chứng: pandas 3.0.3.

**Giáo trình:** McKinney, *Python for Data Analysis*, 3rd ed. — [§5.2](https://wesmckinney.com/book/pandas-basics) (khớp nhãn, áp dụng hàm), [§7.2](https://wesmckinney.com/book/data-cleaning) (tra giá trị bằng map, chia nhóm bằng cut/qcut), [§10.1–10.3](https://wesmckinney.com/book/data-aggregation) (nhóm, tổng hợp, transform, filter), [§8.2–8.3](https://wesmckinney.com/book/data-wrangling) (ghép bảng, đổi hình dạng bảng).

Phần tự học có reindex, chỉ mục lặp, apply theo cột và pivot. Hành vi sửa dữ liệu theo [Copy-on-Write của pandas 3](https://pandas.pydata.org/docs/user_guide/copy_on_write.html).

In [ ]:
import pandas as pd
import numpy as np
print("pandas:", pd.__version__)

### Bài trước

Đã biết chọn hàng, chọn cột, lọc điều kiện và sửa bảng bằng `loc`.

Bài này mở rộng sang phép tính giữa các Series, tính theo nhóm và ghép nhiều bảng.

### Tổng quan

1. Khớp dữ liệu theo nhãn
2. Áp dụng hàm lên dữ liệu
3. Tính theo nhóm
4. Ghép bảng
5. Viết phân tích thành một chuỗi


## 1. Khớp dữ liệu theo nhãn

Tính chênh lệch giá khi hai bảng không cùng thứ tự.

### Tính chênh lệch giá

Hai Series lưu giá qua hai lần thu thập (giả lập, USD/đêm). Chú ý P1 và P2 **đảo thứ tự** ở `gia_moi`:

<div style="display:flex; gap:28px; align-items:flex-start">
<div><b>gia_cu</b>
<table><thead><tr><th>Chỉ mục</th><th>Giá</th></tr></thead><tbody>
<tr style="background:#e5f2f5"><td>P1</td><td>100</td></tr>
<tr style="background:#fcf0df"><td>P2</td><td>80</td></tr></tbody></table></div>
<div><b>gia_moi</b>
<table><thead><tr><th>Chỉ mục</th><th>Giá</th></tr></thead><tbody>
<tr style="background:#fcf0df"><td>P2</td><td>90</td></tr>
<tr style="background:#e5f2f5"><td>P1</td><td>110</td></tr></tbody></table></div>
</div>

```python
gia_moi - gia_cu                        # Trừ hai Series
gia_moi.to_numpy() - gia_cu.to_numpy()  # Bỏ nhãn, trừ hai mảng
```

Hai phép trừ có cho cùng giá trị không? Dự đoán trước khi chạy hai ô dưới.

### Trừ các giá có cùng nhãn

In [ ]:
gia_cu = pd.Series([100, 80], index=["P1", "P2"])
gia_moi = pd.Series([90, 110], index=["P2", "P1"])
print(gia_moi - gia_cu)

In [ ]:
print(gia_moi.to_numpy() - gia_cu.to_numpy())

Series trừ theo **nhãn**; mảng NumPy trừ theo **vị trí**.

### Khi một bên thiếu giá

In [ ]:
gia_moi = pd.Series([90, 110, 70], index=["P2", "P1", "P3"])
print(gia_moi - gia_cu)

Phép trừ lấy **hợp nhãn** của hai Series: kết quả có đủ P1, P2, P3.

P3 chưa có giá cũ nên chưa tính được mức tăng, giảm.

### Đưa mã chỗ ở vào chỉ mục

In [ ]:
phong = pd.DataFrame({"id": ["P1", "P2", "P3"],
                      "gia": [100, 80, 50]})
phong = phong.set_index("id")
print(phong)

`set_index("id")` dùng cột mã làm nhãn của hàng.

`set_index` không kiểm tra mã có lặp hay không: hỏi `phong.index.is_unique`, hoặc bắt pandas báo lỗi ngay bằng `set_index("id", verify_integrity=True)` (xem phần tự học).

### Gán cột cũng khớp theo nhãn

In [ ]:
gia_cap_nhat = pd.Series([90, 110], index=["P2", "P1"])
phong["gia_moi"] = gia_cap_nhat
print(phong)

Phép gán cột căn chỉnh vào các hàng **sẵn có** của bảng, giữ nguyên nhãn của bảng — khác phép trừ ở trên vốn lấy hợp nhãn. P3 không có trong Series nên nhận NaN.

### Bài tập 1: Gán đúng phí

Bảng `phong` dùng mã chỗ ở làm chỉ mục. Phí dọn dẹp (USD) được cung cấp riêng:

```python
ma_cho_o = ["P3", "P1", "P2"]
phi = [20, 10, 30]
```

Thêm cột `phi` vào `phong` sao cho phí luôn khớp với mã chỗ ở, kể cả khi thứ tự hàng của bảng thay đổi.

In [ ]:
# Viết lời giải và kiểm tra kết quả ở đây.

## 2. Áp dụng hàm lên dữ liệu

Tra phí theo loại chỗ ở, rồi tính tiền cho mỗi hàng.

### Bảng chỗ ở và bảng mã phí

Giá tính theo USD/đêm; phí dọn dẹp thu một lần cho mỗi lượt ở. Bảng mã chỉ có hai loại, còn `bang` có ba loại:

<div style="display:flex; gap:28px; align-items:flex-start">
<div><b>bang</b>
<table><thead><tr><th>Chỉ mục</th><th>loai</th><th>gia</th></tr></thead><tbody>
<tr><td>P1</td><td>Nguyên căn</td><td>100</td></tr>
<tr><td>P2</td><td>Phòng riêng</td><td>80</td></tr>
<tr><td>P3</td><td>Khác</td><td>60</td></tr></tbody></table></div>
<div><b>phi_theo_loai</b>
<table><thead><tr><th>loai</th><th>phí</th></tr></thead><tbody>
<tr><td>Nguyên căn</td><td>20</td></tr>
<tr><td>Phòng riêng</td><td>10</td></tr></tbody></table></div>
</div>

Hàng P3 mang loại “Khác”. Cột `phi` của hàng này sẽ nhận giá trị gì?

In [ ]:
bang = pd.DataFrame({
    "loai": ["Nguyên căn", "Phòng riêng", "Khác"],
    "gia": [100, 80, 60],
}, index=["P1", "P2", "P3"])
print(bang)

### Tra phí theo loại chỗ ở

In [ ]:
phi_theo_loai = {"Nguyên căn": 20, "Phòng riêng": 10}
bang["phi"] = bang["loai"].map(phi_theo_loai)
print(bang[["loai", "phi"]])

`map` tra từng giá trị trong bảng mã. “Khác” chưa có mức phí nên nhận NaN. Chưa có mức phí không có nghĩa là miễn phí.

### Viết hàm tính tiền một hàng

Tiền ba đêm = 3 × giá mỗi đêm + phí dọn dẹp.

<table><thead><tr><th>Chỉ mục</th><th>gia</th><th>phi</th></tr></thead><tbody>
<tr><td>P1</td><td>100</td><td>20</td></tr>
<tr><td>P2</td><td>80</td><td>10</td></tr>
<tr><td>P3</td><td>60</td><td>NaN</td></tr></tbody></table>

In [ ]:
def tien_ba_dem(hang):
    return 3 * hang["gia"] + hang["phi"]

### Gọi hàm cho từng hàng

In [ ]:
print(bang.apply(tien_ba_dem, axis=1))

<table><thead><tr><th>Lần gọi</th><th>Giá trị trả về</th></tr></thead><tbody>
<tr><td>P1: 3 × 100 + 20</td><td>320</td></tr>
<tr><td>P2: 3 × 80 + 10</td><td>250</td></tr>
<tr><td>P3: 3 × 60 + NaN</td><td>NaN</td></tr></tbody></table>

`axis=1`: mỗi lần gọi, hàm nhận một hàng dưới dạng Series. `axis=0` nhận từng cột — xem phần tự học.

### Tính trực tiếp trên các cột

In [ ]:
print(3 * bang["gia"] + bang["phi"])

Công thức này viết được bằng phép toán trên cột nên chưa cần `apply`.

### Tải lại bảng Santiago của Bài 4

Từ đây trở đi, các ví dụ trên dữ liệu thật dùng snapshot Santiago ngày 29/06/2026 (giá theo CLP/đêm).

In [ ]:
URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
listings = pd.read_csv(URL)
print(listings.shape)

### So sánh tốc độ hai cách

Cùng một phép tính — tiền cho số đêm tối thiểu — viết bằng `apply(axis=1)` và bằng phép toán trên cột, chạy trên 18.534 hàng.

In [ ]:
def tien_toi_thieu(hang):
    return hang["price"] * hang["minimum_nights"]

In [ ]:
%timeit listings.apply(tien_toi_thieu, axis=1)
%timeit listings["price"] * listings["minimum_nights"]

`apply(axis=1)` dựng một Series Python cho mỗi hàng — đúng khoản chi phí Bài 3 đã chỉ ra khi so vòng `for` với phép tính trên cả mảng. Trên máy giảng viên chênh nhau khoảng 1.000 lần; máy bạn cho con số khác nhưng chênh lệch bậc độ lớn thì giữ nguyên.

### Bài tập 2: Miễn phí dọn dẹp

In [ ]:
dat_cho = pd.DataFrame({"gia": [100, 80, 60], "so_dem": [2, 7, 8],
                         "phi": [20, 10, 15]}, index=["P1", "P2", "P3"])
print(dat_cho)

Bảng `dat_cho`: mỗi hàng là một lượt ở; `gia` tính theo USD/đêm; `so_dem` là số đêm của lượt ở; `phi` là phí dọn dẹp một lần (USD).

<table><thead><tr><th>Chỉ mục</th><th>gia</th><th>so_dem</th><th>phi</th></tr></thead><tbody>
<tr><td>P1</td><td>100</td><td>2</td><td>20</td></tr>
<tr><td>P2</td><td>80</td><td>7</td><td>10</td></tr>
<tr><td>P3</td><td>60</td><td>8</td><td>15</td></tr></tbody></table>

Thêm cột `tong` tính tổng tiền phòng và phí dọn dẹp cho mỗi lượt ở, trong đó khách ở từ 7 đêm được miễn phí dọn dẹp.

In [ ]:
# Viết lời giải và kiểm tra kết quả ở đây.

### Hai cách chia giá thành ba nhóm

`cut` nhận **ngưỡng tuyệt đối** do người phân tích đặt; `qcut` nhận **số nhóm** rồi tự tính ngưỡng phân vị từ dữ liệu. `right=False` để mốc thuộc về nhóm bên phải, khớp mô tả “dưới 30.000 / 30.000–99.999 / từ 100.000”.

In [ ]:
gia = listings["price"].dropna()
nguong_gia = [0, 30000, 100000, np.inf]
nhan = ["thấp", "giữa", "cao"]
cut3 = pd.cut(gia, nguong_gia, right=False, labels=nhan)
qcut3 = pd.qcut(gia, 3, labels=nhan)
print(pd.DataFrame({"cut": cut3.value_counts(),
                    "qcut": qcut3.value_counts()}))

In [ ]:
print(pd.qcut(gia, 3, retbins=True)[1])

`qcut` tự chọn mốc 46.720 và 75.872 để ba nhóm gần bằng nhau.

### Nếu cả thị trường tăng giá 20%

Nhân toàn bộ giá lên 1,2 rồi đếm lại theo đúng hai cách trên.

In [ ]:
gia12 = gia * 1.2
cut12 = pd.cut(gia12, nguong_gia, right=False, labels=nhan)
qcut12 = pd.qcut(gia12, 3, labels=nhan)
print(pd.DataFrame({"cut": cut12.value_counts(),
                    "qcut": qcut12.value_counts()}))

Ngưỡng tuyệt đối so sánh được giữa các mốc chụp và giữa các thành phố. Ngưỡng phân vị khoá cỡ nhóm nên không thấy thị trường lên giá: `qcut` không đổi một dòng nào.

Chọn ranh giới là câu hỏi phân tích, không phải chi tiết kỹ thuật.

## 3. Tính theo nhóm

Từ phép tính trên từng hàng chuyển sang phép tính trên một tập hàng: so sánh giá giữa các khu, rồi so từng chỗ ở với khu của nó.

### So sánh giá của hai khu vực

Bảng giả lập, USD/đêm. Mỗi hàng là một chỗ ở.

In [ ]:
df = pd.DataFrame({
    "id": ["P1", "P2", "P3", "P4", "P5"],
    "khu": ["A", "B", "A", "B", "A"],
    "gia": [100., 80., 200., 120., np.nan],
})
print(df)

### Tính trung bình từng khu

<table><thead><tr><th>khu</th><th>Các giá được nhóm</th><th>Trung bình</th></tr></thead><tbody>
<tr style="background:#e5f2f5"><td>A</td><td>100, 200, NaN</td><td>150</td></tr>
<tr style="background:#fcf0df"><td>B</td><td>80, 120</td><td>100</td></tr></tbody></table>

In [ ]:
print(df.groupby("khu")["gia"].mean())

Nhịp **chia → tính → gộp** (split–apply–combine): `groupby("khu")` chia bảng theo khu, `mean()` tính trên các giá không thiếu của từng nhóm, rồi pandas gộp thành một kết quả cho mỗi khu.

### Đếm số giá dùng để tính

In [ ]:
tk = df.groupby("khu")["gia"].agg(
    trung_binh="mean", so_cho_o="size", so_gia="count")
print(tk)

`size` đếm mọi hàng; `count` chỉ đếm giá không thiếu.

### Đưa tên khu về cột 📖 Tự học

Bảng `tk` đang dùng tên khu làm chỉ mục. Đưa tên khu thành cột để lưu hoặc xử lý tiếp:

In [ ]:
print(tk.reset_index())

### Giá nguyên căn ở Santiago

Dùng bảng `listings` đã tải ở mục 2 (CLP/đêm). So sánh ba khu sau:

In [ ]:
nguyen_can = listings.loc[
    listings["room_type"] == "Entire home/apt"]
thong_ke = nguyen_can.groupby("neighbourhood")["price"].agg(
    trung_vi="median", so_gia="count")
print(thong_ke.loc[["Santiago", "Providencia", "Las Condes"]])

Chỉ xét nguyên căn để tránh gộp chung với phòng riêng. `so_gia` là số giá quan sát được, không phải tổng số chỗ ở.

### Bài tập 3: Trung bình của cả hai khu

Bảng dưới tổng hợp các giá đã biết (USD/đêm).

<table><thead><tr><th>Khu</th><th>Giá trung bình</th><th>Số giá đã biết</th></tr></thead><tbody>
<tr><td>A</td><td>100</td><td>2</td></tr>
<tr><td>B</td><td>200</td><td>8</td></tr></tbody></table>

Từ bảng trên, tính giá trung bình của toàn bộ 10 giá đã biết.

In [ ]:
# Viết lời giải và kiểm tra kết quả ở đây.

### Tách thêm theo loại chỗ ở

Trong mỗi khu, tính riêng giá nguyên căn và phòng riêng.

In [ ]:
df["loai"] = ["Nguyên căn", "Phòng riêng", "Phòng riêng",
              "Nguyên căn", "Nguyên căn"]
print(df.groupby(["khu", "loai"])["gia"].mean())

Mỗi cặp `(khu, loai)` là một nhóm.

### Cùng dữ liệu, hai hình dạng bảng

Kết quả `groupby` theo hai khoá là **dạng dài**: mỗi hàng một cặp `(khu, loại)`.

In [ ]:
dai = listings.groupby(
    ["neighbourhood", "room_type"])["price"].median()
rong = dai.loc[["Santiago", "Providencia", "Las Condes"]]
print(rong.head(4))

In [ ]:
print(rong.unstack("room_type"))

`unstack` đưa một mức chỉ mục lên thành cột. Dạng rộng dễ cho người đọc; dạng dài dễ cho máy vẽ biểu đồ hoặc ghép tiếp.

### Trừ giá cho trung bình của khu

Muốn biết mỗi chỗ ở cao hay thấp hơn trung bình khu của nó. Nhưng `df["gia"]` có chỉ mục 0, 1, 2, 3, 4 còn `tb_khu` có chỉ mục A, B. Dự đoán kết quả trước khi chạy.

In [ ]:
tb_khu = df.groupby("khu")["gia"].mean()
print(df["gia"] - tb_khu)

Hợp nhãn gồm cả 0…4 và A, B; không nhãn nào có mặt ở cả hai bên nên mọi giá trị đều NaN.

### Gắn trung bình khu vào từng hàng

In [ ]:
df["tb_khu"] = df.groupby("khu")["gia"].transform("mean")
df["chenh_lech"] = df["gia"] - df["tb_khu"]
print(df[["id", "khu", "gia", "tb_khu", "chenh_lech"]])

Cùng phép `mean`, khác ở bước gộp: tổng hợp cho **hai** kết quả (một cho mỗi nhóm), `transform("mean")` cho **năm** kết quả (một cho mỗi hàng), giữ nguyên chỉ mục của bảng.

### Giữ lại các khu đủ lớn

Chỉ phân tích các khu có từ 300 chỗ ở trở lên.

In [ ]:
khu_dong = listings.groupby("neighbourhood").filter(
    lambda nhom: len(nhom) >= 300)
print(len(khu_dong), khu_dong["neighbourhood"].nunique())

In [ ]:
mask = listings.groupby("neighbourhood")["id"].transform("size") >= 300
print(listings.loc[mask].equals(khu_dong))

Từ 18.534 hàng thuộc 31 khu còn 16.850 hàng thuộc 8 khu, và hai cách cho đúng cùng một kết quả.

Bộ ba theo nhóm đã đủ: `agg` trả một kết quả cho mỗi nhóm, `transform` trả một kết quả cho mỗi hàng, `filter` giữ lại chính các hàng gốc theo tính chất của cả nhóm.

### So giá từng chỗ ở với khu của nó

In [ ]:
nguyen_can = nguyen_can.copy()
nguyen_can["tv_khu"] = nguyen_can.groupby(
    "neighbourhood")["price"].transform("median")
nguyen_can["lech"] = nguyen_can["price"] - nguyen_can["tv_khu"]
print(nguyen_can[["neighbourhood", "price",
                  "tv_khu", "lech"]].head())

Chỉ mục 2, 4, 5… giữ nguyên từ bảng gốc nên trừ được trực tiếp: mỗi chỗ ở được so với trung vị khu của chính nó. Lệch dương nghĩa là đắt hơn trung vị khu, chưa nói được là đắt xứng đáng hay không.

### Đổi đơn vị quan sát sang host

Mỗi hàng của `listings` là một chỗ ở. Gom theo `host_id` thì mỗi hàng của bảng kết quả là một host.

In [ ]:
host = listings.groupby("host_id").agg(
    so_cho_o=("id", "size"), trung_vi_gia=("price", "median"))
print(host.head())

In [ ]:
print(len(listings), len(host))
print(host["so_cho_o"].median(), host["so_cho_o"].max())
nhieu = host["so_cho_o"] >= 10
print(nhieu.sum(), nhieu.sum() / len(host))
print(host.loc[nhieu, "so_cho_o"].sum(),
      host.loc[nhieu, "so_cho_o"].sum() / len(listings))

18.534 chỗ ở gom lại thành 9.712 host. Một nửa số host chỉ có 1 chỗ ở, host lớn nhất có 253; 198 host (2,0%) nắm 4.337 chỗ ở (23,4%).

Vì thế “giá trung vị của một chỗ ở” và “giá trung vị của một host” là hai con số khác nhau. Mọi KPI phải nói rõ đơn vị đếm.

### Dựng lại một cột của nhà cung cấp

Bảng Santiago có sẵn cột `calculated_host_listings_count`. Tự tính lại rồi đối chiếu.

In [ ]:
n_host = listings.groupby("host_id")["id"].transform("size")
khop = n_host == listings["calculated_host_listings_count"]
print(khop.sum(), len(listings))

Khớp cả 18.534 hàng. Con số do người khác cung cấp nên được dựng lại từ dữ liệu gốc trước khi đưa vào báo cáo.

## 4. Ghép bảng

Từ một bảng chuyển sang nhiều bảng: ghép bảng có thể làm mất hoặc nhân hàng, nên phải kiểm số hàng trước và sau.

### Gộp hai lần thu thập

In [ ]:
dot_1 = pd.DataFrame({"id": ["P1", "P2"], "gia": [100, 80], "dot": 1})
dot_2 = pd.DataFrame({"id": ["P1", "P3"], "gia": [110, 70], "dot": 2})

In [ ]:
print(pd.concat([dot_1, dot_2], ignore_index=True))

`concat` nối thêm hàng. P1 có hai lần thu thập nên xuất hiện hai lần.

### Bổ sung thông tin vùng

In [ ]:
cho_o = pd.DataFrame({"id": ["P1", "P2", "P3"],
                       "khu": ["A", "B", "C"]})
vung = pd.DataFrame({"khu": ["A", "B"], "vung": ["Bắc", "Nam"]})
print(cho_o)
print(vung)

Ghép theo cột `khu`. P3 thuộc C nhưng bảng vùng chưa có C.

### Giữ lại các chỗ ở chưa có vùng

In [ ]:
m = cho_o.merge(vung, on="khu", how="left")
print(m)

`on="khu"`: đối chiếu khu. `how="left"`: giữ mọi hàng của bảng bên trái. Tài liệu pandas gọi cách ghép này là *left join*.

### Tìm hàng không ghép được

Thêm `indicator=True` để biết mỗi chỗ ở có tìm được khu tương ứng trong bảng `vung` hay không.

In [ ]:
kiem_tra = cho_o.merge(vung, on="khu", how="left",
                        indicator=True)
print(kiem_tra)

`both`: tìm thấy khu trong bảng vùng. `left_only`: không tìm thấy — P3 thuộc khu C.

### Khớp khoá mà thiếu thông tin

Lần này bảng vùng **có** khu C, nhưng ô `vung` của C đang trống. Trước khi chạy, dự đoán `_merge` của P3.

In [ ]:
vung_thieu = pd.DataFrame({"khu": ["A", "B", "C"],
                           "vung": ["Bắc", "Nam", np.nan]})
print(cho_o.merge(vung_thieu, on="khu", how="left", indicator=True))

Cả ba hàng đều `both`: khoá C có khớp, chỉ là giá trị vùng chưa biết. NaN ở cột bên phải **không** đồng nghĩa với không khớp khoá — muốn biết hàng nào không khớp thì đọc cột `_merge`, đừng đếm NaN.

### Chỉ lấy các chỗ ở ghép được vùng

In [ ]:
print(cho_o.merge(vung, on="khu", how="inner"))

`how="inner"` (*inner join*) chỉ giữ các cặp khớp khoá. P3 không có trong kết quả. Mặc định của `merge` là `inner`, nên quên `how` có thể làm mất hàng.

### Khi bảng tra cứu lặp khoá

In [ ]:
vung_lap = pd.DataFrame({"khu": ["A", "A", "B"],
                         "vung": ["Bắc", "Trung tâm", "Nam"]})
print(cho_o.merge(vung_lap, on="khu", how="left"))

P1 khớp hai hàng mang khoá A nên xuất hiện hai lần.

### Báo lỗi nếu bảng vùng lặp khu

`many_to_one`: nhiều chỗ ở có thể cùng khu; bảng vùng chỉ được có một dòng cho mỗi khu.

In [ ]:
try:
    cho_o.merge(vung_lap, on="khu", how="left",
                validate="many_to_one")
except pd.errors.MergeError as err:
    print(type(err).__name__, str(err))

Khu A lặp trong bảng vùng nên pandas dừng phép ghép thay vì lặng lẽ nhân đôi P1.

### Ghép bảng host trở lại bảng chỗ ở

Bảng `host` dựng ở mục 3 có khoá `host_id` duy nhất, nên mỗi chỗ ở chỉ khớp đúng một host.

In [ ]:
mo_rong = listings.merge(host, on="host_id", how="left",
                         validate="many_to_one")
assert len(mo_rong) == len(listings)
print(len(listings), "->", len(mo_rong))

`validate` bảo đảm mỗi chỗ ở chỉ khớp một host; dòng `assert` ghi lại kỳ vọng số hàng không đổi để lần chạy sau tự báo lỗi nếu dữ liệu thay đổi. Mỗi lần ghép trong bài tập lớn nên có một dòng như vậy.

### Bài tập 4: Ghép hai bảng

<div style="display:flex; gap:28px; align-items:flex-start">
<div><b>trai</b>
<table><thead><tr><th>id</th><th>khu</th></tr></thead><tbody>
<tr><td>P1</td><td>A</td></tr><tr><td>P2</td><td>A</td></tr><tr><td>P3</td><td>B</td></tr></tbody></table></div>
<div><b>phai</b>
<table><thead><tr><th>khu</th><th>vung</th></tr></thead><tbody>
<tr><td>A</td><td>Bắc</td></tr><tr><td>A</td><td>Trung tâm</td></tr></tbody></table></div>
</div>

Viết bảng kết quả của lệnh sau:

```python
trai.merge(phai, on="khu", how="left")
```

In [ ]:
trai = pd.DataFrame({"id": ["P1", "P2", "P3"], "khu": ["A", "A", "B"]})
phai = pd.DataFrame({"khu": ["A", "A"], "vung": ["Bắc", "Trung tâm"]})

# Ghi kết quả dự đoán, rồi chạy lệnh để kiểm tra.

## 5. Viết phân tích thành một chuỗi

Từ nhiều ô gán chồng lên bảng chuyển sang một chuỗi trả bảng mới: chạy lại bao nhiêu lần cũng ra một kết quả.


### Chạy lại một ô có đổi kết quả không?

Một ô tăng giá 10%. Chạy ô thứ hai chính là chạy lại đúng dòng đó lần nữa.

In [ ]:
bang_gia = pd.DataFrame({"gia": [100.0, 80.0]}, index=["P1", "P2"])
bang_gia["gia"] = bang_gia["gia"] * 1.1
print(bang_gia)

In [ ]:
bang_gia["gia"] = bang_gia["gia"] * 1.1   # chạy lại đúng dòng trên
print(bang_gia)

Phép gán chồng lên chính cột cũ nên chạy hai lần ra kết quả khác. Nhìn vào bảng thì không biết ô đã chạy mấy lần.

Cách chữa tạm thời khi vẫn muốn gán: luôn tính từ cột gốc, ví dụ `bang_gia["gia_moi"] = bang_gia["gia"] * 1.1`.

### Đặt tên cho từng bước

Mỗi bước là một hàm nhận bảng và trả về một cột mới, không sửa bảng. Hai hàm dưới dùng lại đúng `cut` và `transform` đã học; tên hàm nói rõ bước đó làm gì.


In [ ]:
def chia_phan_khuc(d):
    return pd.cut(d["price"], nguong_gia, right=False, labels=nhan)

def dem_host(d):
    return d.groupby("host_id")["id"].transform("size")


### Thêm cột, trả bảng mới

Dự đoán trước khi chạy: `mo_rong` có bao nhiêu cột so với `co_gia`? `listings` có thêm cột `phan_khuc` không?


In [ ]:
co_gia = listings.loc[listings["price"].notna()]
mo_rong = co_gia.assign(phan_khuc=chia_phan_khuc, n_host=dem_host)
print(co_gia.shape, mo_rong.shape)


In [ ]:
print("phan_khuc" in listings.columns)


`assign` nhận hàm, gọi hàm trên bảng hiện tại và trả một bảng mới có thêm cột. Bảng `listings` ban đầu không đổi, nên chạy ô trên bao nhiêu lần cũng chỉ tạo bảng mới giống nhau. Khác với `bang_gia["gia"] = ...` ở trên: không có phép gán vào bảng gốc.


### Ghép các bước thành một chuỗi

Mỗi dòng là một bước: lọc → thêm cột → nhóm → tổng hợp.


In [ ]:
ket_qua = (listings
    .loc[listings["price"].notna()]
    .assign(phan_khuc=chia_phan_khuc, n_host=dem_host)
    .groupby("phan_khuc", observed=True)
    .agg(so_cho_o=("id", "size"),
         trung_vi_gia=("price", "median"),
         tv_n_host=("n_host", "median")))
print(ket_qua)


Chuỗi không sửa `listings`, nên chạy lại vẫn ra đúng một kết quả — điều kiện để bài tập lớn chạy lại từ đầu đến cuối bằng một lệnh, và là cách tránh đúng cái bẫy “cột dẫn xuất không tự tính lại” của Bài 4: mọi cột đều tính lại từ đầu mỗi lần chạy.

`observed=True` chỉ giữ các nhóm thực sự có dữ liệu. Đọc kết quả: chỗ ở càng đắt càng hay thuộc về host nhiều chỗ ở (trung vị 1 → 2 → 3) — đây là quan sát, chưa phải kết luận nhân quả.


## Tự học theo giáo trình

Các ví dụ sau mở rộng phần đã học; không cần dùng để giải các bài tập phía trên.

### Xếp lại theo danh sách nhãn

Dự đoán giá trị tại P3 trước khi chạy. `reindex` sắp theo nhãn yêu cầu; nhãn chưa có nhận giá trị thiếu.

In [ ]:
print(gia_cu.reindex(["P2", "P3", "P1"]))

### Chỉ mục có thể lặp

Một nhãn có thể chọn nhiều hàng. Nếu yêu cầu mỗi mã xác định một hàng, cần kiểm tra tính duy nhất.

In [ ]:
lap = pd.Series([10, 20, 30], index=["a", "a", "b"])
print(lap.loc["a"])
print(lap.index.is_unique)

Khi đưa cột mã vào chỉ mục, `verify_integrity=True` yêu cầu mã không lặp.

In [ ]:
du_lieu = pd.DataFrame({"id": ["P1", "P1"], "gia": [100, 110]})
try:
    du_lieu.set_index("id", verify_integrity=True)
except ValueError as err:
    print(type(err).__name__, str(err))

### Áp dụng hàm cho từng cột

Với `axis=0` (mặc định), `apply` gọi hàm một lần cho **mỗi cột**, và hàm nhận cả cột dưới dạng Series. So số lần gọi với `axis=1` ở phần chính.

In [ ]:
def mo_ta(cot):
    return f"{cot.name}: {cot.count()} giá trị"

print(bang[["gia", "phi"]].apply(mo_ta))

### Tra trung bình khu bằng map

Một Series cũng có thể làm bảng tra cho `map`. So kết quả dưới đây với cột `tb_khu` từ `transform`.

In [ ]:
tb_theo_khu = df.groupby("khu")["gia"].mean()
print(df["khu"].map(tb_theo_khu))
print(df["khu"].map(tb_theo_khu).equals(df["tb_khu"]))

### Đưa loại chỗ ở thành cột bằng pivot_table

`pivot_table` gộp hai bước lại: vừa nhóm và tổng hợp như `groupby`, vừa sắp nhóm lên hàng và cột như `unstack`. Mặc định `aggfunc="mean"`, và cũng bỏ qua giá trị thiếu giống `groupby`.

So kết quả dưới đây với `groupby` theo hai khoá: các nhóm được sắp trên hàng và cột như thế nào?

In [ ]:
print(df.pivot_table(index="khu", columns="loai", values="gia", aggfunc="mean"))

### Làm việc với AI thì sao?

**🤖 AI làm tốt** — viết code tính toán, nhóm và ghép từ yêu cầu cụ thể.

**⚠️ AI hay sai** — bỏ nhãn khi tính, bỏ sót điều kiện, chọn sai khoá nhóm, không kiểm tra các hàng bị lặp sau ghép, và trả bảng đã đổi đơn vị đếm mà không nói.

### Kiểm chứng thế nào?

1. Thử dữ liệu đảo thứ tự, thiếu nhãn và giá trị đúng biên điều kiện.
2. Tính tay một nhóm; kiểm tra số giá tham gia phép tính.
3. Dự đoán số hàng sau ghép rồi kiểm tra khoá và các hàng chưa khớp.
4. Hỏi mỗi hàng của bảng kết quả là gì: một chỗ ở, một host hay một lượt thu thập?

### Đọc thêm

1. [McKinney, §5.2: khớp nhãn và áp dụng hàm](https://wesmckinney.com/book/pandas-basics)
2. [McKinney, chương 10: tính theo nhóm](https://wesmckinney.com/book/data-aggregation)
3. [McKinney, §8.2: ghép dữ liệu; §8.3: đổi hình dạng bảng](https://wesmckinney.com/book/data-wrangling)
4. [pandas: kiểm tra phép ghép](https://pandas.pydata.org/docs/user_guide/merging.html)

---

## Tóm tắt bài học

- Phép tính và phép gán Series khớp theo nhãn.
- Dùng hàm hoặc phép toán cột để thực hiện đúng quy tắc tính.
- Tổng hợp trả kết quả cho mỗi nhóm; `transform` trả kết quả cho từng hàng.
- Ghép bảng có thể làm mất hoặc nhân hàng, nên phải kiểm số hàng trước và sau.
- Viết phân tích thành một chuỗi trả bảng mới để chạy lại luôn ra cùng kết quả.

Chạy **Restart & Run all** để kiểm tra từ đầu. Khi dùng AI, đối chiếu bằng bảng nhỏ tính tay được.

**Bài sau:** đọc và truy xuất dữ liệu từ file, API và SQL.